# Week 95, Day 3: Writing the test

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('wraptext.py', 'w', encoding='utf-8') as f:
    f.write('"""Shorten text to fit a width.\n\n>>> shorten("Hello world, nice day", 18)\n\'Hello world, [...]\'\n"""\n\n\ndef shorten(text, width, placeholder=" [...]"):\n    """Collapse whitespace and cut text to fit width, ending with placeholder if anything was cut."""\n    words = text.split()\n    joined = " ".join(words)\n    if len(joined) <= width:\n        return joined\n    kept, length = [], 0\n    for word in words:\n        extra = len(word) + (1 if kept else 0)\n        if length + extra + len(placeholder) > width:\n            break\n        kept.append(word)\n        length += extra\n    if not kept:\n        raise ValueError("placeholder too large for max width")\n    return " ".join(kept) + placeholder\n')

**Predict before you run:** what will this print?



In [ ]:
import unittest

class TestShorten(unittest.TestCase):
    def test_fits(self):
        self.assertEqual(len("Hello world, [...]"), 18)

    def check_spaces(self):
        self.assertEqual(1, 2)

result = unittest.TestResult()
unittest.defaultTestLoader.loadTestsFromTestCase(TestShorten).run(result)
print(result.testsRun, len(result.failures))

> **Key idea.** Test in CPython's style: `unittest.TestCase`, `test_` methods, `assertEqual`, `assertRaises` as a context manager, `subTest` for many cases, next to the existing tests. Write the regression test from the minimal case first and see it fail on the old code. Test the boundaries: exact fit, one either side, already fits, whitespace, too-long words, tiny widths. Measure the suite by the mutants it kills; watch for tests that never run, tests without assertions, and `assertTrue(a == b)`.

### Exercise: The regression test

Write a `unittest` test class for `shorten` from `wraptext.py` that passes on the fixed module and fails on the original bug and on a version that allows one character too many.

In [ ]:
import unittest
from wraptext import shorten

class TestShorten(unittest.TestCase):
    def test_fits_already(self):
        self.assertEqual(shorten("Hello world", 40), "Hello world")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"wraptext.py\").read()\nBROKEN = {\n    \"the original bug\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) >= width\"),\n    \"one too many\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) > width + 1\"),\n    \"no collapsing\": GOOD.replace(\"if len(joined) <= width\", \"if len(text) <= width\"),\n    \"no error\": GOOD.replace('raise ValueError(\"placeholder too large for max width\")', \"return placeholder.strip()\"),\n    \"placeholder ignored\": GOOD.replace(\"length + extra + len(placeholder)\", \"length + extra\"),\n    \"words run together\": GOOD.replace('return \" \".join(kept) + placeholder', 'return \"\".join(kept) + placeholder'),\n}\nWANTED = {k: BROKEN[k] for k in (\"the original bug\", \"one too many\")}\n\ndef test_suite():\n    \"the bug, and its neighbour\"\n    for name, code in WANTED.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"wraptext\", {\"correct\": GOOD, **WANTED})\n    write_file(\"wraptext.py\", GOOD)\n    missed = [k for k in WANTED if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Testing the boundaries

Extend the tests so they catch all six broken versions: the original bug, one character too many, no whitespace collapsing, no error for a width too small, the placeholder ignored when measuring, and kept words run together. Use `subTest` for at least one group of cases.

In [ ]:
import unittest
from wraptext import shorten

class TestShorten(unittest.TestCase):
    def test_exact_fit_is_kept(self):
        self.assertEqual(shorten("Hello a nice day today", 18), "Hello a nice [...]")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"wraptext.py\").read()\nBROKEN = {\n    \"the original bug\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) >= width\"),\n    \"one too many\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) > width + 1\"),\n    \"no collapsing\": GOOD.replace(\"if len(joined) <= width\", \"if len(text) <= width\"),\n    \"no error\": GOOD.replace('raise ValueError(\"placeholder too large for max width\")', \"return placeholder.strip()\"),\n    \"placeholder ignored\": GOOD.replace(\"length + extra + len(placeholder)\", \"length + extra\"),\n    \"words run together\": GOOD.replace('return \" \".join(kept) + placeholder', 'return \"\".join(kept) + placeholder'),\n}\n\ndef test_suite():\n    \"all six caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    assert \"subTest\" in source, \"use subTest for a group of cases\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"wraptext\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"wraptext.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Running a test class

Write `run_suite(case_class)`: load the class's tests with `unittest.defaultTestLoader.loadTestsFromTestCase`, run them into a `unittest.TestResult`, and return `(run, problems, names)`: the number of tests run, the number of failures plus errors, and the sorted method names of the tests that failed or errored (each test's `_testMethodName`).

In [ ]:
import unittest

def run_suite(case_class):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import unittest\n\nclass Sample(unittest.TestCase):\n    def test_ok(self):\n        self.assertEqual(1, 1)\n    def test_wrong(self):\n        self.assertEqual(1, 2)\n    def test_crash(self):\n        {}[\"missing\"]\n    def helper(self):\n        self.fail(\"never runs\")\n\ndef test_run():\n    \"counts and names\"\n    assert run_suite(Sample) == (3, 2, [\"test_crash\", \"test_wrong\"])")

### Exercise: Test smells

Write `find_smells(source)`: for every method of every class in the source whose base list includes a name or attribute ending in `TestCase`, report in source order: `"LINE NAME: never runs"` for a method not starting with `test` or `_` that calls an `assert...` method of `self`; `"LINE NAME: no assertion"` for a `test` method with no call to a `self.assert...` or `self.fail` method and no `assert` statement; and `"LINE NAME: use assertEqual"` (with the line of the call) for each `self.assertTrue(...)` whose argument is an `==` comparison.

In [ ]:
import ast

def find_smells(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SOURCE = '''import unittest\nfrom wraptext import shorten\n\nclass TestShorten(unittest.TestCase):\n    def test_exact_fit(self):\n        self.assertTrue(shorten(\"Hello a nice day today\", 18) == \"Hello a nice [...]\")\n\n    def check_spaces(self):\n        self.assertEqual(shorten(\"a  b\", 3), \"a b\")\n\n    def test_runs(self):\n        shorten(\"Hello\", 3)\n\n    def _helper(self):\n        return 1\n\n    def test_error(self):\n        with self.assertRaises(ValueError):\n            shorten(\"one two\", 3)\n'''\n\ndef test_found():\n    \"three smells\"\n    assert find_smells(SOURCE) == [\"6 test_exact_fit: use assertEqual\", \"8 check_spaces: never runs\", \"11 test_runs: no assertion\"]\n    assert find_smells(\"class Plain:\\n    def check(self):\\n        self.assertEqual(1, 1)\\n\") == []")

### Bug hunt: A test that never ran

The test method's name doesn't start with `test`, so the suite runs nothing and prints `0 run, 0 failed`. Rename it so it runs, and prints `1 run, 0 failed`.

In [ ]:
import unittest
from wraptext import shorten

class TestShorten(unittest.TestCase):
    def check_exact_fit(self):
        self.assertEqual(shorten("Hello a nice day today", 18), "Hello a nice [...]")

result = unittest.TestResult()
unittest.defaultTestLoader.loadTestsFromTestCase(TestShorten).run(result)
print(f"{result.testsRun} run, {len(result.failures)} failed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"test_ methods run\"\n    expect_output(\"1 run, 0 failed\")")

### Bug hunt: A test that can't fail

`assertTrue(x, msg)` only checks that `x` is truthy (the second argument is a message), so this test passes even against the buggy version defined here, and prints `0 failed`. Make it compare the values, so it catches the bug and prints `1 failed`.

In [ ]:
import unittest

def shorten(text, width, placeholder=" [...]"):
    return "Hello a [...]"

class TestShorten(unittest.TestCase):
    def test_exact_fit(self):
        self.assertTrue(shorten("Hello a nice day today", 18), "Hello a nice [...]")

result = unittest.TestResult()
unittest.defaultTestLoader.loadTestsFromTestCase(TestShorten).run(result)
print(f"{len(result.failures)} failed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"assertEqual\"\n    expect_output(\"1 failed\")")

### Puzzle: Many cases, one test

Arrange a test that checks several widths with `subTest`, so each failing width is reported separately. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the folder where CPython's tests live (with its slash)
q = ___   # the unittest method that checks an exception is raised
r = ___   # True or False: a regression test should pass on the unfixed code
s = ___   # what a mutation that no test catches is called (one word)
t = ___   # the prefix a TestCase method needs to run

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"Lib/test/\" and q == \"assertRaises\" and r is False, \"p, q and r\"\n    assert s == \"survivor\" or s == \"surviving\", \"s\"\n    assert t == \"test\", \"t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: your own mutation tester

Write `mutation_score(test_source, module_name, good, mutants)` without pytest. For each version (the good source, then each mutant in order): create a module with `types.ModuleType(module_name)`, execute the version's source in its `__dict__`, put it in `sys.modules[module_name]`, execute `test_source` in a fresh namespace, and run every `unittest.TestCase` subclass defined there with `unittest.TestResult`. A version is **caught** if any test fails or errors. Restore `sys.modules` afterwards (remove the entry if it wasn't there). Return lines: `"tests: N"` (tests run on the good version), `"on the fix: all pass"` or `"on the fix: K fail"`, `"caught: ..."` and `"missed: ..."` (mutant names, or `none`), and `"score: P%"` (caught over mutants, rounded).

In [ ]:
import sys
import types
import unittest

def mutation_score(test_source, module_name, good, mutants):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"wraptext.py\").read()\nBROKEN = {\n    \"the original bug\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) >= width\"),\n    \"one too many\": GOOD.replace(\"len(placeholder) > width\", \"len(placeholder) > width + 1\"),\n    \"no collapsing\": GOOD.replace(\"if len(joined) <= width\", \"if len(text) <= width\"),\n    \"no error\": GOOD.replace('raise ValueError(\"placeholder too large for max width\")', \"return placeholder.strip()\"),\n    \"placeholder ignored\": GOOD.replace(\"length + extra + len(placeholder)\", \"length + extra\"),\n    \"words run together\": GOOD.replace('return \" \".join(kept) + placeholder', 'return \"\".join(kept) + placeholder'),\n}\n\nTESTS = '''import unittest\nfrom wraptext import shorten\n\nclass TestShorten(unittest.TestCase):\n    def test_exact_fit_is_kept(self):\n        self.assertEqual(shorten(\"Hello a nice day today\", 18), \"Hello a nice [...]\")\n\n    def test_one_short(self):\n        self.assertEqual(shorten(\"Hello a nice day today\", 17), \"Hello a [...]\")\n'''\n\ndef test_score():\n    \"two tests, six mutants\"\n    lines = mutation_score(TESTS, \"wraptext\", GOOD, BROKEN)\n    assert lines == [\"tests: 2\", \"on the fix: all pass\", \"caught: the original bug, one too many, placeholder ignored, words run together\",\n                     \"missed: no collapsing, no error\", \"score: 67%\"]\n\ndef test_restores():\n    \"sys.modules is left as it was\"\n    import sys\n    before = sys.modules.get(\"wraptext\")\n    mutation_score(TESTS, \"wraptext\", GOOD, {})\n    assert sys.modules.get(\"wraptext\") is before\n    bad_tests = TESTS.replace('\"Hello a [...]\")', '\"nope\")')\n    assert mutation_score(bad_tests, \"wraptext\", GOOD, {})[1] == \"on the fix: 1 fail\"")

## Warm-up

Only methods whose names start with `test` run. `check_spaces` would fail, but it never runs, so the suite looks greener than it is. That's one of the test smells you'll learn to spot today. `wraptext.py` on this page now has the fix from yesterday.

## Tests in CPython's style

CPython's tests live in `Lib/test/test_NAME.py` and use `unittest`: a class derived from `unittest.TestCase`, methods named `test_...`, and assertion methods that give helpful messages when they fail: `assertEqual(a, b)`, `assertTrue(x)`, `assertIn(a, b)`, `assertRaises(Error)` used as a context manager (`with self.assertRaises(ValueError): ...`). For many similar cases in one method, `with self.subTest(text=text, width=width):` reports each failing case separately instead of stopping at the first. Add your test next to the existing tests for the same function, in the same style, rather than in a new file. pytest runs `unittest` classes too, which is how this page checks yours.

## The regression test

A **regression test** reproduces the reported bug: it uses Day 1's minimal case and asserts the correct result. Write it **first**, run it against the unfixed code and watch it **fail** (if it passes, it doesn't test the bug), then apply the fix and watch it pass. CPython tests often mention the issue in a comment (`# gh-140123: an exact fit is kept`) so the next reader knows why it exists.

## Boundaries

Bugs cluster at edges. For `shorten`: text that exactly fills the width, one character more and one fewer; text that fits without shortening; repeated spaces; a single word too long for the width; a width too small for even the placeholder; a different placeholder. One test per behaviour, each with a name that says what it checks (`test_exact_fit_is_kept`), is easier to read when it fails than one test that checks everything.

## Is the test any good?

A test suite is good if it **fails when the code is wrong**. Week 24's mutation testing measures that directly: make small plausible mistakes in the code (a **mutant**: `>` to `>=`, removing the whitespace collapse) and check the suite catches each one. A surviving mutant points at a missing test. Smells to look for too: test methods that never run, tests with no assertion, and `assertTrue(a == b)` where `assertEqual(a, b)` would say what the values were.

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Write the regression test first** for your issue, in the existing test file, and run it on unfixed `main`: it must fail.
2. **Apply your fix** and run it again (`./python -m test test_NAME -v`): it must pass.
3. **Add boundary tests** for the edges near your bug, using `subTest` where cases are alike.
4. **Mutate your own fix** by hand (undo it, or make it off by one) and check your tests catch each version.

## Before you go

Tomorrow you'll open the pull request: a commit message, a description, a NEWS entry and a careful look at your own diff before anyone else sees it.